# See where floating point and high precision mislead

*Adversarial Mathematics From First Principles*, [Part II, chapter 4](https://programmer.ie/books/math/13-chapter/) · [worked example](https://programmer.ie/examples/math/13-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

Ordinary computer arithmetic rounds, and higher precision only rounds later.
This program shows three places where that changes an answer: 0.1 + 0.2, the
parity of the Fermat number F₆ = 2⁶⁴ + 1 once it has been converted to a
float, and e^(π√163), which looks like an integer until 40 digits. It ends
with the arithmetic behind Simpson's paradox: a combined rate is a weighted
average of the group rates.

## Run it

Standard library only; nothing to install.

In [1]:
"""Exact numbers and misleading aggregates (standard library only)."""
from decimal import Decimal, getcontext
from fractions import Fraction as F

print("Floating point:")
print(f"  0.1 + 0.2 == 0.3 -> {0.1 + 0.2 == 0.3}  (stored as {0.1 + 0.2!r})")
print(f"  as fractions     -> {F(1, 10) + F(2, 10) == F(3, 10)}")
f6 = 2 ** 64 + 1
print(f"  F6 = 2^64 + 1 = {f6}")
print(f"  float(F6) % 2 = {float(f6) % 2}   (the float is exactly 2^64: 'even')")
print(f"  F6 % 2        = {f6 % 2}     (exact: odd)")
print(f"  F6 == 274177 * 67280421310721: {f6 == 274177 * 67280421310721}")
print(f"  (10**16 + 1) - 10**16 as floats: {(1e16 + 1) - 1e16}, as integers: {(10**16 + 1) - 10**16}")


def pi_decimal(digits):
    """pi by Machin's formula, to the current Decimal precision."""
    getcontext().prec = digits + 10

    def arctan_inv(n):
        x, n2, total, k, sign = Decimal(1) / n, n * n, Decimal(0), 1, 1
        term = x
        while term:
            total += sign * term / k
            x /= n2
            term, k, sign = x, k + 2, -sign
        return total

    return 16 * arctan_inv(5) - 4 * arctan_inv(239)


print("\nHigh precision is still not exact: exp(pi * sqrt(163))")
for digits in (15, 30, 40):
    getcontext().prec = digits + 10
    value = (pi_decimal(digits) * Decimal(163).sqrt()).exp()
    getcontext().prec = digits
    print(f"  {digits} digits: {+value}")
print("  At 15 or 30 digits it looks like an integer; at 40 it is ...743.99999999999925: not an integer.")

print("\nA combined rate is the mediant, a weighted average of the group rates (open surgery, kidney stones):")
small, large = F(81, 87), F(192, 263)
combined = F(81 + 192, 87 + 263)
print(f"  small 81/87 = {float(small):.3f}, large 192/263 = {float(large):.3f}, combined 273/350 = {float(combined):.3f}")
print(f"  combined == 87/350 * small + 263/350 * large: {combined == F(87, 350) * small + F(263, 350) * large}")
print(f"  lies between the group rates: {min(small, large) < combined < max(small, large)}")

Floating point:
  0.1 + 0.2 == 0.3 -> False  (stored as 0.30000000000000004)
  as fractions     -> True
  F6 = 2^64 + 1 = 18446744073709551617
  float(F6) % 2 = 0.0   (the float is exactly 2^64: 'even')
  F6 % 2        = 1     (exact: odd)
  F6 == 274177 * 67280421310721: True
  (10**16 + 1) - 10**16 as floats: 0.0, as integers: 1

High precision is still not exact: exp(pi * sqrt(163))
  15 digits: 2.62537412640769E+17
  30 digits: 262537412640768743.999999999999
  40 digits: 262537412640768743.9999999999992500725972
  At 15 or 30 digits it looks like an integer; at 40 it is ...743.99999999999925: not an integer.

A combined rate is the mediant, a weighted average of the group rates (open surgery, kidney stones):
  small 81/87 = 0.931, large 192/263 = 0.730, combined 273/350 = 0.780
  combined == 87/350 * small + 263/350 * large: True
  lies between the group rates: True


## What the output shows

The float conversion happens once, and every later step inherits it: F₆
becomes 2⁶⁴, which is even. Exact integers keep it odd, and Landry's
factorization checks exactly. At 15 or 30 digits e^(π√163) gives no reason to
doubt that it is an integer; at 40 it misses by about 7.5 × 10⁻¹³. Agreement
to many digits is numerical evidence, not proof. The kidney-stone rate 273/350
sits between its group rates, weighted by where the patients were.

## Not shown

- The π used for e^(π√163) is computed by Machin's formula to the working precision; the digits shown are as reliable as that computation, not proved.
- The mediant identity is checked for one arm of one table, not proved in general.

## Challenge it

Examine where each computation in the program loses or keeps information:
the float conversion of 2⁶⁴ + 1, the 15-, 30- and 40-digit values of
e^(π√163), and the combined kidney-stone rate. Prove that a mediant lies
between the fractions it combines, and find two groups and weights for which
a treatment better in both groups is worse combined.